# Genie Space Configuration — TPC-H Analytics

This notebook is a **modular configuration tool** for managing the Genie space. Edit the configuration cells below, then run the execution cells to create or update the space.

## Notebook Structure
| Cell | Purpose | Action |
|---|---|---|
| **Cells 2-7** | Configuration (edit these) | Define space settings, instructions, metric views, questions, SQLs, benchmarks |
| **Cell 8** | Helper functions | Builds the `serialized_space` JSON from your config |
| **Cell 9** | Create / Update space | Calls the Genie API to apply changes |
| **Cell 10** | Validate space | Reads back the space and displays a summary |

## How to Use
1. Edit any configuration cell (Cells 2–7)
2. Run **Cell 8** (helpers), then **Cell 9** (apply), then **Cell 10** (validate)
3. To create a **new** space: set `SPACE_ID = None` in Cell 2
4. To **update** an existing space: set `SPACE_ID = "<existing_id>"` in Cell 2

In [0]:
# =============================================================================
# SPACE CONFIGURATION
# Edit these values to configure the Genie space.
# =============================================================================

SPACE_TITLE = "TPC-H Analytics Genie"

SPACE_DESCRIPTION = (
    "Explore TPC-H order analytics using natural language. Ask questions about "
    "revenue, orders, customers, suppliers, shipping patterns, and market segments "
    "across global regions and nations. Powered by a centralized metric view built "
    "on the TPC-H lineitem fact table with snowflake schema joins to orders, "
    "customers, suppliers, nations, and regions."
)

# Set to None to CREATE a new space, or an existing ID to UPDATE
SPACE_ID = ""
WAREHOUSE_ID = "<your-sql-warehouse-id>"
PARENT_PATH = "/Users/your.name@company.com"

print(f"Space: {SPACE_TITLE}")
print(f"Mode:  {'UPDATE existing' if SPACE_ID else 'CREATE new'}")


Space: TPC-H Analytics Genie
Mode:  CREATE new


In [0]:
# =============================================================================
# GENERAL INSTRUCTIONS
# This is the single instruction block that teaches Genie about the space.
# Edit the text below to change how Genie interprets questions.
# The API allows exactly ONE text_instructions entry.
# =============================================================================

GENERAL_INSTRUCTIONS = """
This Genie space provides natural-language access to TPC-H order analytics — revenue, orders, customers, suppliers, shipping patterns, and market segments across global regions and nations. It is powered by 1 metric view with 5 measures and 13 dimensions built on the TPC-H lineitem fact table with snowflake schema joins.

METRIC VIEW: aira_test.aibi_workshop_schema.order_management_metrics
- Source: samples.tpch.lineitem (fact table)
- Joins: orders → customer → customer_nation → customer_region; supplier → supplier_nation → supplier_region
- Dimensions (13): Order Month, Order Date, Order Status, Order Priority, Customer Name, Market Segment, Customer Nation, Customer Region, Supplier Name, Supplier Nation, Supplier Region, Ship Mode, Ship Instruction
- Measures (5): Revenue, Line Item Count, Order Count, Unique Customers, Avg Delivery Days

ORDER STATUS REFERENCE: O = Open, P = Processing, F = Fulfilled. The metric view already maps these codes to readable labels.

ORDER PRIORITY VALUES: 1-URGENT, 2-HIGH, 3-MEDIUM, 4-NOT SPECIFIED, 5-LOW.

MARKET SEGMENTS: AUTOMOBILE, BUILDING, FURNITURE, HOUSEHOLD, MACHINERY.

SHIP MODES: AIR, FOB, MAIL, RAIL, REG AIR, SHIP, TRUCK.

REGIONS: AFRICA, AMERICA, ASIA, EUROPE, MIDDLE EAST.

QUERY PATTERNS:
- Always use MEASURE() syntax for measures: MEASURE(`Revenue`), MEASURE(`Order Count`), etc.
- Use backtick-quoted column names for dimensions: `Market Segment`, `Customer Region`, etc.
- Group by dimensions using GROUP BY ALL for convenience.
- For time-series analysis, use `Order Month` for monthly trends and `Order Date` for daily granularity.
- Revenue is calculated as SUM(l_extendedprice * (1 - l_discount)) — it represents net revenue after discounts.
- Avg Delivery Days is AVG(DATEDIFF(l_receiptdate, l_shipdate)) — the average days from shipment to receipt.
""".strip()

print(f"Instructions length: {len(GENERAL_INSTRUCTIONS)} chars")
print(f"Preview: {GENERAL_INSTRUCTIONS[:100]}...")

Instructions length: 1825 chars
Preview: This Genie space provides natural-language access to TPC-H order analytics — revenue, orders, custom...


In [0]:
# =============================================================================
# METRIC VIEW DESCRIPTIONS
# Map fully-qualified metric view names to their descriptions.
# Keys MUST be sorted alphabetically (the helper function enforces this).
# =============================================================================

METRIC_VIEW_DESCRIPTIONS = {
    "aira_test.aibi_workshop_schema.order_management_metrics": (
        "Centralized TPC-H order analytics metric view. "
        "13 dimensions covering order attributes (month, date, status, priority), "
        "customer attributes (name, market segment, nation, region), "
        "supplier attributes (name, nation, region), and shipping attributes (mode, instruction). "
        "5 measures: Revenue (net after discount), Line Item Count, Order Count, "
        "Unique Customers, and Avg Delivery Days. "
        "Source: samples.tpch.lineitem with snowflake joins to orders, customer, supplier, nation, region."
    ),
}

print(f"Metric views configured: {len(METRIC_VIEW_DESCRIPTIONS)}")
for name in METRIC_VIEW_DESCRIPTIONS:
    print(f"  • {name.split('.')[-1]}")

# Validate alphabetical order
keys = list(METRIC_VIEW_DESCRIPTIONS.keys())
assert keys == sorted(keys), "ERROR: Metric view keys must be sorted alphabetically!"
print("\n✅ Keys are sorted alphabetically")

Metric views configured: 1
  • order_management_metrics

✅ Keys are sorted alphabetically


In [0]:
# =============================================================================
# SAMPLE QUESTIONS
# These appear in the Genie chat UI as suggested questions.
# Add, remove, or edit questions as needed.
# =============================================================================

SAMPLE_QUESTIONS = [
    # --- Revenue ---
    "What is total revenue by customer region?",
    "Show me monthly revenue trend",
    "Which market segment generates the most revenue?",
    "What is revenue by ship mode?",
    # --- Orders ---
    "How many orders are in each status?",
    "Show me order count by order priority",
    "What is the monthly order volume trend?",
    "Which customer region has the most orders?",
    # --- Customers ---
    "How many unique customers do we have by market segment?",
    "Show me the top 10 customers by revenue",
    "What is the customer distribution by nation?",
    # --- Suppliers ---
    "Which supplier region generates the most revenue?",
    "Show me top 10 suppliers by line item count",
    "What is supplier distribution by nation?",
    # --- Shipping ---
    "What is the average delivery time by ship mode?",
    "Show me delivery days trend by month",
    "Which ship mode has the most line items?",
    "What is average delivery time by customer region?",
    # --- Cross-domain ---
    "Compare revenue across regions and market segments",
    "Show me fulfilled vs open orders by region",
    "What is revenue by order priority and market segment?",
    "Which nations have the highest average delivery days?",
]

print(f"Sample questions configured: {len(SAMPLE_QUESTIONS)}")


Sample questions configured: 22


In [0]:
# =============================================================================
# EXAMPLE QUESTION SQLS
# These teach Genie HOW to answer questions (part of instructions).
# Each tuple is (question_text, sql_query).
# Add, remove, or edit pairs as needed.
# =============================================================================

C = "aira_test.aibi_workshop_schema"  # shorthand for readability

EXAMPLE_QUESTION_SQLS = [
    # --- Revenue ---
    (
        "What is total revenue by customer region?",
        f"SELECT `Customer Region`, MEASURE(`Revenue`) AS revenue, MEASURE(`Order Count`) AS order_count, MEASURE(`Unique Customers`) AS unique_customers FROM {C}.order_management_metrics GROUP BY ALL ORDER BY revenue DESC"
    ),
    (
        "Show me monthly revenue trend",
        f"SELECT `Order Month`, MEASURE(`Revenue`) AS revenue, MEASURE(`Order Count`) AS order_count, MEASURE(`Line Item Count`) AS line_items FROM {C}.order_management_metrics GROUP BY ALL ORDER BY `Order Month`"
    ),
    (
        "Which market segment generates the most revenue?",
        f"SELECT `Market Segment`, MEASURE(`Revenue`) AS revenue, MEASURE(`Order Count`) AS order_count, MEASURE(`Unique Customers`) AS unique_customers FROM {C}.order_management_metrics GROUP BY ALL ORDER BY revenue DESC"
    ),
    (
        "What is revenue by ship mode?",
        f"SELECT `Ship Mode`, MEASURE(`Revenue`) AS revenue, MEASURE(`Line Item Count`) AS line_items, MEASURE(`Avg Delivery Days`) AS avg_delivery_days FROM {C}.order_management_metrics GROUP BY ALL ORDER BY revenue DESC"
    ),
    # --- Orders ---
    (
        "How many orders are in each status?",
        f"SELECT `Order Status`, MEASURE(`Order Count`) AS order_count, MEASURE(`Revenue`) AS revenue, MEASURE(`Line Item Count`) AS line_items FROM {C}.order_management_metrics GROUP BY ALL ORDER BY order_count DESC"
    ),
    (
        "Show me order count by order priority",
        f"SELECT `Order Priority`, MEASURE(`Order Count`) AS order_count, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY `Order Priority`"
    ),
    (
        "What is the monthly order volume trend?",
        f"SELECT `Order Month`, MEASURE(`Order Count`) AS order_count, MEASURE(`Line Item Count`) AS line_items, MEASURE(`Unique Customers`) AS unique_customers FROM {C}.order_management_metrics GROUP BY ALL ORDER BY `Order Month`"
    ),
    (
        "Which customer region has the most orders?",
        f"SELECT `Customer Region`, MEASURE(`Order Count`) AS order_count, MEASURE(`Revenue`) AS revenue, MEASURE(`Unique Customers`) AS unique_customers FROM {C}.order_management_metrics GROUP BY ALL ORDER BY order_count DESC"
    ),
    # --- Customers ---
    (
        "How many unique customers do we have by market segment?",
        f"SELECT `Market Segment`, MEASURE(`Unique Customers`) AS unique_customers, MEASURE(`Order Count`) AS order_count, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY unique_customers DESC"
    ),
    (
        "Show me the top 10 customers by revenue",
        f"WITH ranked AS (SELECT `Customer Name`, MEASURE(`Revenue`) AS revenue, MEASURE(`Order Count`) AS order_count, ROW_NUMBER() OVER (ORDER BY MEASURE(`Revenue`) DESC) AS rn FROM {C}.order_management_metrics GROUP BY ALL) SELECT `Customer Name`, revenue, order_count FROM ranked WHERE rn <= 10 ORDER BY revenue DESC"
    ),
    (
        "What is the customer distribution by nation?",
        f"SELECT `Customer Nation`, `Customer Region`, MEASURE(`Unique Customers`) AS unique_customers, MEASURE(`Order Count`) AS order_count, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY unique_customers DESC"
    ),
    # --- Suppliers ---
    (
        "Which supplier region generates the most revenue?",
        f"SELECT `Supplier Region`, MEASURE(`Revenue`) AS revenue, MEASURE(`Line Item Count`) AS line_items, MEASURE(`Avg Delivery Days`) AS avg_delivery_days FROM {C}.order_management_metrics GROUP BY ALL ORDER BY revenue DESC"
    ),
    (
        "Show me top 10 suppliers by line item count",
        f"WITH ranked AS (SELECT `Supplier Name`, MEASURE(`Line Item Count`) AS line_items, MEASURE(`Revenue`) AS revenue, ROW_NUMBER() OVER (ORDER BY MEASURE(`Line Item Count`) DESC) AS rn FROM {C}.order_management_metrics GROUP BY ALL) SELECT `Supplier Name`, line_items, revenue FROM ranked WHERE rn <= 10 ORDER BY line_items DESC"
    ),
    (
        "What is supplier distribution by nation?",
        f"SELECT `Supplier Nation`, `Supplier Region`, MEASURE(`Line Item Count`) AS line_items, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY line_items DESC"
    ),
    # --- Shipping ---
    (
        "What is the average delivery time by ship mode?",
        f"SELECT `Ship Mode`, MEASURE(`Avg Delivery Days`) AS avg_delivery_days, MEASURE(`Line Item Count`) AS line_items, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY avg_delivery_days"
    ),
    (
        "Show me delivery days trend by month",
        f"SELECT `Order Month`, MEASURE(`Avg Delivery Days`) AS avg_delivery_days, MEASURE(`Line Item Count`) AS line_items FROM {C}.order_management_metrics GROUP BY ALL ORDER BY `Order Month`"
    ),
    (
        "Which ship mode has the most line items?",
        f"SELECT `Ship Mode`, MEASURE(`Line Item Count`) AS line_items, MEASURE(`Revenue`) AS revenue, MEASURE(`Order Count`) AS order_count FROM {C}.order_management_metrics GROUP BY ALL ORDER BY line_items DESC"
    ),
    (
        "What is average delivery time by customer region?",
        f"SELECT `Customer Region`, MEASURE(`Avg Delivery Days`) AS avg_delivery_days, MEASURE(`Line Item Count`) AS line_items, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY avg_delivery_days"
    ),
    # --- Cross-domain ---
    (
        "Compare revenue across regions and market segments",
        f"SELECT `Customer Region`, `Market Segment`, MEASURE(`Revenue`) AS revenue, MEASURE(`Order Count`) AS order_count FROM {C}.order_management_metrics GROUP BY ALL ORDER BY `Customer Region`, revenue DESC"
    ),
    (
        "Show me fulfilled vs open orders by region",
        f"SELECT `Customer Region`, `Order Status`, MEASURE(`Order Count`) AS order_count, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY `Customer Region`, `Order Status`"
    ),
    (
        "What is revenue by order priority and market segment?",
        f"SELECT `Order Priority`, `Market Segment`, MEASURE(`Revenue`) AS revenue, MEASURE(`Order Count`) AS order_count FROM {C}.order_management_metrics GROUP BY ALL ORDER BY `Order Priority`, revenue DESC"
    ),
    (
        "Which nations have the highest average delivery days?",
        f"SELECT `Customer Nation`, `Customer Region`, MEASURE(`Avg Delivery Days`) AS avg_delivery_days, MEASURE(`Line Item Count`) AS line_items FROM {C}.order_management_metrics GROUP BY ALL ORDER BY avg_delivery_days DESC LIMIT 10"
    ),
]

print(f"Example question SQLs configured: {len(EXAMPLE_QUESTION_SQLS)}")

Example question SQLs configured: 22


In [0]:
# =============================================================================
# BENCHMARK QUESTIONS
# These EVALUATE Genie accuracy (separate from instructions).
# Each tuple is (question_text, expected_sql_answer).
# Genie does NOT learn from benchmarks — they are for testing only.
# =============================================================================

BENCHMARK_QUESTIONS = [
    # --- Revenue ---
    (
        "What is total revenue by customer region?",
        f"SELECT `Customer Region`, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY revenue DESC"
    ),
    (
        "Which market segment has the highest revenue?",
        f"SELECT `Market Segment`, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY revenue DESC LIMIT 1"
    ),
    (
        "Show revenue by ship mode",
        f"SELECT `Ship Mode`, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY revenue DESC"
    ),
    (
        "What is the monthly revenue trend?",
        f"SELECT `Order Month`, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY `Order Month`"
    ),
    # --- Orders ---
    (
        "How many orders are open vs fulfilled?",
        f"SELECT `Order Status`, MEASURE(`Order Count`) AS order_count FROM {C}.order_management_metrics GROUP BY ALL ORDER BY order_count DESC"
    ),
    (
        "Show order count by priority",
        f"SELECT `Order Priority`, MEASURE(`Order Count`) AS order_count FROM {C}.order_management_metrics GROUP BY ALL ORDER BY `Order Priority`"
    ),
    (
        "What is the total number of line items?",
        f"SELECT MEASURE(`Line Item Count`) AS total_line_items FROM {C}.order_management_metrics"
    ),
    (
        "How many orders per month?",
        f"SELECT `Order Month`, MEASURE(`Order Count`) AS order_count FROM {C}.order_management_metrics GROUP BY ALL ORDER BY `Order Month`"
    ),
    # --- Customers ---
    (
        "How many unique customers by region?",
        f"SELECT `Customer Region`, MEASURE(`Unique Customers`) AS unique_customers FROM {C}.order_management_metrics GROUP BY ALL ORDER BY unique_customers DESC"
    ),
    (
        "Show top 10 customers by revenue",
        f"WITH ranked AS (SELECT `Customer Name`, MEASURE(`Revenue`) AS revenue, ROW_NUMBER() OVER (ORDER BY MEASURE(`Revenue`) DESC) AS rn FROM {C}.order_management_metrics GROUP BY ALL) SELECT `Customer Name`, revenue FROM ranked WHERE rn <= 10 ORDER BY revenue DESC"
    ),
    (
        "What is unique customer count by market segment?",
        f"SELECT `Market Segment`, MEASURE(`Unique Customers`) AS unique_customers FROM {C}.order_management_metrics GROUP BY ALL ORDER BY unique_customers DESC"
    ),
    (
        "Show customer count by nation",
        f"SELECT `Customer Nation`, MEASURE(`Unique Customers`) AS unique_customers FROM {C}.order_management_metrics GROUP BY ALL ORDER BY unique_customers DESC"
    ),
    # --- Suppliers ---
    (
        "Which supplier region has the most revenue?",
        f"SELECT `Supplier Region`, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY revenue DESC"
    ),
    (
        "Show top 10 suppliers by revenue",
        f"WITH ranked AS (SELECT `Supplier Name`, MEASURE(`Revenue`) AS revenue, ROW_NUMBER() OVER (ORDER BY MEASURE(`Revenue`) DESC) AS rn FROM {C}.order_management_metrics GROUP BY ALL) SELECT `Supplier Name`, revenue FROM ranked WHERE rn <= 10 ORDER BY revenue DESC"
    ),
    (
        "How many line items by supplier nation?",
        f"SELECT `Supplier Nation`, MEASURE(`Line Item Count`) AS line_items FROM {C}.order_management_metrics GROUP BY ALL ORDER BY line_items DESC"
    ),
    # --- Shipping ---
    (
        "What is average delivery days by ship mode?",
        f"SELECT `Ship Mode`, MEASURE(`Avg Delivery Days`) AS avg_delivery_days FROM {C}.order_management_metrics GROUP BY ALL ORDER BY avg_delivery_days"
    ),
    (
        "Which ship mode has the most line items?",
        f"SELECT `Ship Mode`, MEASURE(`Line Item Count`) AS line_items FROM {C}.order_management_metrics GROUP BY ALL ORDER BY line_items DESC"
    ),
    (
        "Show average delivery days by customer region",
        f"SELECT `Customer Region`, MEASURE(`Avg Delivery Days`) AS avg_delivery_days FROM {C}.order_management_metrics GROUP BY ALL ORDER BY avg_delivery_days"
    ),
    # --- Cross-domain ---
    (
        "Compare revenue by region and market segment",
        f"SELECT `Customer Region`, `Market Segment`, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY `Customer Region`, revenue DESC"
    ),
    (
        "Show fulfilled orders by region",
        f"SELECT `Customer Region`, MEASURE(`Order Count`) AS order_count, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics WHERE `Order Status` = 'Fulfilled' GROUP BY ALL ORDER BY order_count DESC"
    ),
    (
        "What is revenue by priority and region?",
        f"SELECT `Order Priority`, `Customer Region`, MEASURE(`Revenue`) AS revenue FROM {C}.order_management_metrics GROUP BY ALL ORDER BY `Order Priority`, revenue DESC"
    ),
    (
        "Which nations have the slowest delivery?",
        f"SELECT `Customer Nation`, `Customer Region`, MEASURE(`Avg Delivery Days`) AS avg_delivery_days FROM {C}.order_management_metrics GROUP BY ALL ORDER BY avg_delivery_days DESC LIMIT 10"
    ),
]

print(f"Benchmark questions configured: {len(BENCHMARK_QUESTIONS)}")

Benchmark questions configured: 22


In [0]:
# =============================================================================
# HELPER FUNCTIONS
# Run this cell before Cell 9 (Create/Update) or Cell 10 (Validate).
# =============================================================================

import json
import uuid
import requests


def get_workspace_url():
    """Get the current workspace URL."""
    return spark.conf.get("spark.databricks.workspaceUrl")


def get_api_headers():
    """Get authorization headers for the Genie API."""
    token = (
        dbutils.notebook.entry_point.getDbutils()
        .notebook().getContext().apiToken().get()
    )
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }


def _sorted_hex_ids(n: int) -> list[str]:
    """Generate *n* sorted 32-char lowercase hex UUIDs."""
    return sorted(uuid.uuid4().hex for _ in range(n))


def build_serialized_space(
    general_instructions: str,
    metric_view_descriptions: dict[str, str],
    sample_questions: list[str],
    example_question_sqls: list[tuple[str, str]],
    benchmark_questions: list[tuple[str, str]],
) -> str:
    """
    Build the serialized_space JSON string from user-editable config.

    Returns a JSON string ready for the Genie Space API.
    All IDs are auto-generated, sorted, and guaranteed unique.
    """

    # ---- Generate sorted IDs for every section ----
    sq_ids = _sorted_hex_ids(len(sample_questions))
    eq_ids = _sorted_hex_ids(len(example_question_sqls))
    bm_ids = _sorted_hex_ids(len(benchmark_questions))
    ti_id  = uuid.uuid4().hex                           # single instruction

    # Verify global uniqueness
    all_ids = sq_ids + eq_ids + bm_ids + [ti_id]
    assert len(all_ids) == len(set(all_ids)), "UUID collision — rerun the cell."

    # ---- config.sample_questions ----
    config_sq = [
        {"id": sq_ids[i], "question": [q]}
        for i, q in enumerate(sample_questions)
    ]

    # ---- data_sources.metric_views (sorted by identifier) ----
    mv_list = [
        {"identifier": k, "description": [v]}
        for k, v in sorted(metric_view_descriptions.items())
    ]

    # ---- instructions.text_instructions (exactly one entry) ----
    text_instr = [{"id": ti_id, "content": [general_instructions]}]

    # ---- instructions.example_question_sqls (sorted by id) ----
    ex_sqls = [
        {"id": eq_ids[i], "question": [q], "sql": [sql]}
        for i, (q, sql) in enumerate(example_question_sqls)
    ]

    # ---- benchmarks.questions (sorted by id) ----
    bm_list = [
        {
            "id": bm_ids[i],
            "question": [q],
            "answer": [{"format": "SQL", "content": [sql]}],
        }
        for i, (q, sql) in enumerate(benchmark_questions)
    ]

    # ---- Assemble full structure ----
    payload = {
        "version": 2,
        "config": {"sample_questions": config_sq},
        "data_sources": {"metric_views": mv_list},
        "instructions": {
            "text_instructions": text_instr,
            "example_question_sqls": ex_sqls,
        },
        "benchmarks": {"questions": bm_list},
    }

    return json.dumps(payload)


print("✅ Helper functions loaded: get_workspace_url, get_api_headers, build_serialized_space")


✅ Helper functions loaded: get_workspace_url, get_api_headers, build_serialized_space


In [0]:
# =============================================================================
# CREATE OR UPDATE GENIE SPACE
# Run this cell to apply the configuration from Cells 2–7.
# =============================================================================

serialised = build_serialized_space(
    general_instructions=GENERAL_INSTRUCTIONS,
    metric_view_descriptions=METRIC_VIEW_DESCRIPTIONS,
    sample_questions=SAMPLE_QUESTIONS,
    example_question_sqls=EXAMPLE_QUESTION_SQLS,
    benchmark_questions=BENCHMARK_QUESTIONS,
)

ws_url  = get_workspace_url()
headers = get_api_headers()

if SPACE_ID:
    # ---------- UPDATE existing space ----------
    print(f"Updating space {SPACE_ID} ...")
    resp = requests.patch(
        f"https://{ws_url}/api/2.0/genie/spaces/{SPACE_ID}",
        headers=headers,
        json={
            "title": SPACE_TITLE,
            "description": SPACE_DESCRIPTION,
            "serialized_space": serialised,
        },
    )
else:
    # ---------- CREATE new space ----------
    print("Creating new space ...")
    resp = requests.post(
        f"https://{ws_url}/api/2.0/genie/spaces",
        headers=headers,
        json={
            "title": SPACE_TITLE,
            "description": SPACE_DESCRIPTION,
            "warehouse_id": WAREHOUSE_ID,
            "parent_path": PARENT_PATH,
            "serialized_space": serialised,
        },
    )

# ---------- Report result ----------
if resp.status_code == 200:
    result = resp.json()
    new_id = result.get("space_id", SPACE_ID)
    print(f"\n✅ SUCCESS")
    print(f"   Space ID   : {new_id}")
    print(f"   Title       : {result.get('title')}")
    print(f"   Description : {result.get('description', '')[:120]}...")
    if not SPACE_ID:
        print(f"\n⚠️  Copy this ID into Cell 2 for future updates:")
        print(f'   SPACE_ID = "{new_id}"')
else:
    print(f"\n❌ FAILED ({resp.status_code})")
    err = resp.json()
    print(f"   Error : {err.get('error_code')}")
    print(f"   Message: {err.get('message', '')[:300]}")


Creating new space ...

✅ SUCCESS
   Space ID   : 01f13125c7e2196b99fba750e225cdb8
   Title       : TPC-H Analytics Genie
   Description : Explore TPC-H order analytics using natural language. Ask questions about revenue, orders, customers, suppliers, shippin...

⚠️  Copy this ID into Cell 2 for future updates:
   SPACE_ID = "01f13125c7e2196b99fba750e225cdb8"


In [0]:
# =============================================================================
# VALIDATE SPACE
# Read back the space config and display a summary.
# =============================================================================

target_id = SPACE_ID
if not target_id:
    print("No SPACE_ID set — set it in Cell 2 after creating a space.")
else:
    ws_url  = get_workspace_url()
    headers = get_api_headers()

    resp = requests.get(
        f"https://{ws_url}/api/2.0/genie/spaces/{target_id}"
        "?include_serialized_space=true",
        headers=headers,
    )

    if resp.status_code != 200:
        print(f"❌ Failed to read space: {resp.status_code}")
    else:
        data = resp.json()
        ss   = json.loads(data["serialized_space"])

        sqs  = ss.get("config", {}).get("sample_questions", [])
        mvs  = ss.get("data_sources", {}).get("metric_views", [])
        tis  = ss.get("instructions", {}).get("text_instructions", [])
        eqs  = ss.get("instructions", {}).get("example_question_sqls", [])
        bms  = ss.get("benchmarks", {}).get("questions", [])

        print("=" * 60)
        print("GENIE SPACE VALIDATION REPORT")
        print("=" * 60)
        print(f"Title        : {data.get('title')}")
        print(f"Space ID     : {data.get('space_id', target_id)}")
        print(f"Warehouse    : {data.get('warehouse_id', 'N/A')}")
        print(f"Description  : {data.get('description', '')[:120]}...")
        print()
        print(f"Metric Views        : {len(mvs)}")
        for mv in mvs:
            print(f"   • {mv['identifier'].split('.')[-1]}")
        print(f"Sample Questions    : {len(sqs)}")
        print(f"Example SQLs        : {len(eqs)}")
        print(f"Benchmark Questions : {len(bms)}")
        print(f"Text Instructions   : {len(tis)} block(s), "
              f"{sum(len(t['content'][0]) for t in tis)} chars")
        print()

        if bms:
            print("-" * 60)
            print("BENCHMARK QUESTIONS")
            print("-" * 60)
            for i, bm in enumerate(bms, 1):
                q = bm['question'][0]
                has_sql = bool(bm.get('answer'))
                marker = "✅" if has_sql else "⚠️"
                print(f"  {i:2d}. {marker} {q}")

        print()
        print("✅ Validation complete")


No SPACE_ID set — set it in Cell 2 after creating a space.
